In [ ]:
import asyncio
import logging

import pandas as pd
from tqdm.asyncio import tqdm
from vero.traces.analysis import TraceAnalyzer, plot_session_scores_with_table
from vero_benchmarking.constants import DEFAULT_RESULTS_DIR
from vero_benchmarking.utils import get_path_to_vero_agents

logging.getLogger("vero.git").setLevel(logging.WARNING)


df = pd.read_csv(DEFAULT_RESULTS_DIR / "benchmark_results.csv", index_col=0)
project_path = get_path_to_vero_agents()
analyzer = TraceAnalyzer()

In [ ]:
async def batch_analyze_sessions(
    session_ids: list[str],
    project_path: str,
    analyzer: TraceAnalyzer | None = None,
    max_concurrency: int = 5,
) -> dict[str, list[str]]:
    if analyzer is None:
        analyzer = TraceAnalyzer()

    semaphore = asyncio.Semaphore(max_concurrency)

    async def analyze_session_wrapper(session_id: str):
        async with semaphore:
            try:
                _, _ = await analyzer.analyze_session(
                    session_id=session_id,
                    project_path=project_path,
                    show_progress=False,
                    return_payload=True,
                    save_to_cache=True,
                    use_cache=True,
                )
                return session_id
            except Exception as e:
                return e

    status_dict = {"success": [], "error": []}

    analysis_coros = [analyze_session_wrapper(session_id) for session_id in session_ids]
    results = await tqdm.gather(*analysis_coros)

    for session_id, result in zip(session_ids, results):
        if isinstance(result, Exception):
            status_dict["error"].append(session_id)
            print(f"Error analyzing session {session_id}: {result}")
        else:
            status_dict["success"].append(session_id)

    return status_dict


session_ids = df.session_id.tolist()
status_dict = await batch_analyze_sessions(session_ids, project_path)  # noqa: F704
status_dict

In [ ]:
row_idxs = [80]


def print_commits(payload):
    for i in range(len(payload.phases)):
        print([x.message for x in payload.phases[i].commits])


for row_idx in row_idxs:
    row = df.iloc[row_idx]
    session_id = row.session_id
    payload, analysis = await analyzer.analyze_session(  # noqa: F704
        session_id=session_id,
        project_path=project_path,
        return_payload=True,
        save_to_cache=False,
        use_cache=True,
    )

    print(f"Session {session_id}")
    print(f"Base commit: {payload.config.base_commit}")
    print(f"Final commit: {payload.config.final_commit}")
    print_commits(payload)
    print()
    print()

In [ ]:
fig = plot_session_scores_with_table(analysis, title=payload.config.base_branch)
fig.show()